# Qualidade e preparação dos dados

Limpa as 5 tabelas usadas e gera três arquivos em `data/processed/`:
uma linha por pedido, uma linha por cliente e os pedidos com o
identificador do cliente. Cada filtro tem contagem e justificativa em
[docs/decisions.md](../docs/decisions.md).

## Carga e validações

Carrega as tabelas, converte as datas (erro se algo não converter) e
valida as chaves.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

INICIO = pd.Timestamp("2017-01-01")
FIM = pd.Timestamp("2018-08-31")
LIMITE = pd.Timestamp("2018-03-04")

def ler(nome, datas=()):
    df = pd.read_csv(RAW / f"olist_{nome}_dataset.csv")
    for col in datas:
        df[col] = pd.to_datetime(df[col])  # erro se algo não converter
    return df

o = ler("orders", ["order_purchase_timestamp",
                   "order_delivered_customer_date",
                   "order_estimated_delivery_date"])
c = ler("customers")
it = ler("order_items")
pg = ler("order_payments")
rv = ler("order_reviews", ["review_creation_date", "review_answer_timestamp"])

assert o["order_id"].is_unique
assert c["customer_id"].is_unique
assert o["customer_id"].isin(c["customer_id"]).all()
assert rv["review_score"].between(1, 5).all()

## Como medir o atraso

A data estimada vem sempre às 00:00, enquanto a entrega tem hora. Por
isso o atraso é calculado comparando só as datas.

In [ ]:
est = o["order_estimated_delivery_date"]
ent = o["order_delivered_customer_date"].dropna()
print("estimada com hora 00:00:", (est.dt.hour == 0).mean())
print("entrega com hora 00:00:", (ent.dt.hour == 0).mean())

## Pedidos válidos

Só pedidos entregues, com data de entrega e entre 2017-01 e 2018-08. A
tabela abaixo mostra quantos pedidos cada regra remove.

In [ ]:
log = []

def aplicar(df, nome, mascara):
    depois = df[mascara]
    log.append({"regra": nome, "antes": len(df), "depois": len(depois),
                "removidos": len(df) - len(depois)})
    return depois

p = o.copy()
p = aplicar(p, "status = delivered", p["order_status"] == "delivered")
p = aplicar(p, "com data de entrega", p["order_delivered_customer_date"].notna())
p = aplicar(p, "entrega >= compra",
            p["order_delivered_customer_date"] >= p["order_purchase_timestamp"])
p = aplicar(p, "compra entre 2017-01-01 e 2018-08-31",
            p["order_purchase_timestamp"].dt.normalize().between(INICIO, FIM))
pd.DataFrame(log)

## Uma linha por pedido

Itens, pagamentos e avaliações são agregados antes de juntar, e o
`validate` nos joins faz o pandas dar erro se alguma junção duplicar
linhas. A soma de preços é conferida contra a tabela original.

In [ ]:
itens = (it.groupby("order_id")
           .agg(n_itens=("order_item_id", "count"),
                valor_itens=("price", "sum"),
                frete=("freight_value", "sum"))
           .reset_index())
pagos = (pg.groupby("order_id")
           .agg(valor_pago=("payment_value", "sum"))
           .reset_index())
nota = (rv.sort_values(["order_id", "review_creation_date", "review_answer_timestamp"])
          .drop_duplicates("order_id", keep="last")[["order_id", "review_score"]]
          .rename(columns={"review_score": "nota"}))

pedidos = (p[["order_id", "customer_id", "order_purchase_timestamp",
              "order_delivered_customer_date", "order_estimated_delivery_date"]]
           .rename(columns={"order_purchase_timestamp": "compra",
                            "order_delivered_customer_date": "entrega",
                            "order_estimated_delivery_date": "estimada"})
           .merge(itens, on="order_id", how="left", validate="1:1")
           .merge(pagos, on="order_id", how="left", validate="1:1")
           .merge(nota, on="order_id", how="left", validate="1:1"))

assert pedidos["order_id"].is_unique
soma_original = it[it["order_id"].isin(pedidos["order_id"])]["price"].sum()
assert np.isclose(pedidos["valor_itens"].sum(), soma_original)
print(len(pedidos))
print(pedidos[["n_itens", "valor_pago", "nota"]].isna().sum())

## Atraso

Atraso = data de entrega posterior à data estimada.

In [ ]:
pedidos["dias_atraso"] = (pedidos["entrega"].dt.normalize()
                          - pedidos["estimada"].dt.normalize()).dt.days
pedidos["atrasado"] = pedidos["dias_atraso"] > 0
print(pedidos["atrasado"].mean())
print(pedidos["dias_atraso"].describe())

## Uma linha por cliente

Para cada cliente: data do primeiro pedido, se o primeiro dia teve algum
pedido atrasado e se houve recompra em 1 a 180 dias. Pedidos feitos no
mesmo dia do primeiro contam como a mesma compra. A base elegível tem
primeiro pedido até 2018-03-04, para todos terem 180 dias de observação.

In [ ]:
cli = pedidos.merge(c[["customer_id", "customer_unique_id"]],
                    on="customer_id", how="left", validate="m:1")
assert cli["customer_unique_id"].notna().all()

cli["data"] = cli["compra"].dt.normalize()
cli["primeira_data"] = cli.groupby("customer_unique_id")["data"].transform("min")
cli["dias_desde_primeiro"] = (cli["data"] - cli["primeira_data"]).dt.days

primeiro_dia = cli[cli["dias_desde_primeiro"] == 0]
clientes = (primeiro_dia.groupby("customer_unique_id")
            .agg(primeira_data=("data", "min"),
                 atrasado_primeiro=("atrasado", "max"),
                 pedidos_primeiro_dia=("order_id", "count")))

rec = (cli[cli["dias_desde_primeiro"].between(1, 180)]
       .groupby("customer_unique_id").size().rename("pedidos_180d"))
clientes = clientes.join(rec)
clientes["pedidos_180d"] = clientes["pedidos_180d"].fillna(0).astype(int)
clientes["recompra_180"] = clientes["pedidos_180d"] > 0
clientes["elegivel"] = clientes["primeira_data"].between(INICIO, LIMITE)

## Conferência e saída

Tamanho da base elegível e dos dois grupos (a comparação fica para o
notebook `02_analysis`). Os arquivos são salvos em `data/processed/`,
que não é versionada.

In [ ]:
base = clientes[clientes["elegivel"]]
print("clientes totais:", len(clientes), " elegíveis:", len(base))
print("taxa de recompra (elegíveis):", base["recompra_180"].mean())
print(base.groupby("atrasado_primeiro")["recompra_180"].agg(["size", "sum"]))

pedidos.to_parquet(PROCESSED / "pedidos.parquet", index=False)
clientes.to_parquet(PROCESSED / "clientes.parquet")
cli[["order_id", "customer_unique_id", "data", "primeira_data",
     "dias_desde_primeiro", "atrasado"]].to_parquet(
    PROCESSED / "pedidos_cliente.parquet", index=False)